# Casos reales: seguimiento de esferas en un ensayo de impacto

Pipeline completo para los dos ensayos (caucho y hielo), fusionado en un solo notebook:

1. **Detección** de la esfera en cada fotograma del vuelo libre (sustracción de fondo + contorno para el caucho / transformada de Hough para el hielo, con una segunda pasada de radio fijo).
2. **Velocidad**: ajuste robusto de la trayectoria, paso de píxeles a m/s con el diámetro real de la esfera y la frecuencia de captura de la cámara.
3. **Comparación** de escala propia frente a escala común.
4. **Gráficas** de trayectoria y ajuste.

Antes de ejecutar, ajusta las rutas `RUTA_ENTRADA` y `RUTA_SALIDA` de la primera celda.

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
#  RUTAS  (ajustar a tu equipo)
# ============================================================
# RUTA_ENTRADA debe contener las carpetas Test_Caucho/ y Test_Hielo/,
# cada una con los fotogramas frame_XXXX.png del vuelo libre.
RUTA_ENTRADA = "datos/Fotogramas"
# RUTA_SALIDA es donde se guardan las detecciones, los CSV y las graficas.
RUTA_SALIDA = "resultados"

# --- Ensayos: ventana de vuelo libre y metodo de deteccion ---
ensayos = {
    "Caucho": {"frame_ini": 1, "frame_fin": 69, "metodo": "disco",  "umbral": 25},
    "Hielo":  {"frame_ini": 1, "frame_fin": 67, "metodo": "anillo", "umbral": 18},
}

# --- Parametros de deteccion ---
radio_min = 28
radio_max = 45
circularidad_min = 0.55
area_min = 120
tam_apertura = 3
dilatacion_anillo = 3
soporte_min = 0.35
grosor_anillo = 4

# --- Segunda pasada (radio fijo) ---
refinar = True
ventana_x = 22
ventana_y = 8
soporte_min_refinado = 0.38

# --- Datos fisicos del ensayo ---
diametro_real_mm = 30.0
fps_real = 20000                                   # fps de la camara de alta velocidad
referencia = {"Caucho": 108.0, "Hielo": 109.0}     # m/s, valor de referencia (dato)

# --- Filtros de fiabilidad para la velocidad ---
radio_min_vel = 20.0
radio_max_vel = 60.0
ancho_imagen = 1024
k_outlier = 2.5

In [ ]:
# ============================================================
#  Funciones de deteccion
# ============================================================

def cargar_frames(ruta, f_ini, f_fin):
    nombres = sorted(f for f in os.listdir(ruta) if f.endswith(".png"))
    seleccion = []
    for archivo in nombres:
        numero = int(archivo.replace("frame_", "").replace(".png", ""))
        if f_ini <= numero <= f_fin:
            seleccion.append((numero, archivo))
    return seleccion


def construir_fondo(ruta, seleccion):
    pila = []
    for _, archivo in seleccion:
        imagen = cv2.imread(os.path.join(ruta, archivo))
        pila.append(cv2.cvtColor(imagen, cv2.COLOR_BGR2GRAY))
    mediana = np.median(np.stack(pila, axis=0), axis=0)
    return mediana.astype(np.uint8)


def mascara_movimiento(gris, fondo, umbral_diff):
    diferencia = cv2.absdiff(gris, fondo)
    _, mascara = cv2.threshold(diferencia, umbral_diff, 255, cv2.THRESH_BINARY)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (tam_apertura, tam_apertura))
    mascara = cv2.morphologyEx(mascara, cv2.MORPH_OPEN, kernel)
    mascara = cv2.morphologyEx(mascara, cv2.MORPH_CLOSE, kernel)
    return mascara


def detectar_disco(mascara):
    contornos, _ = cv2.findContours(mascara, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    mejor, mejor_puntuacion = None, -1.0
    for contorno in contornos:
        area = cv2.contourArea(contorno)
        if area < area_min:
            continue
        perimetro = cv2.arcLength(contorno, True)
        if perimetro == 0:
            continue
        circularidad = 4.0 * np.pi * area / (perimetro * perimetro)
        if circularidad < circularidad_min:
            continue
        (x, y), radio = cv2.minEnclosingCircle(contorno)
        if not (radio_min <= radio <= radio_max):
            continue
        puntuacion = circularidad * area
        if puntuacion > mejor_puntuacion:
            mejor_puntuacion = puntuacion
            mejor = (float(x), float(y), float(radio))
    return mejor


def soporte_anillo(mascara, x, y, radio):
    anillo = np.zeros_like(mascara)
    cv2.circle(anillo, (int(x), int(y)), int(radio), 255, grosor_anillo)
    total = cv2.countNonZero(anillo)
    if total == 0:
        return 0.0
    apoyado = cv2.countNonZero(cv2.bitwise_and(anillo, mascara))
    return apoyado / total


def detectar_anillo(mascara):
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (dilatacion_anillo, dilatacion_anillo))
    unida = cv2.dilate(mascara, kernel, iterations=1)
    suave = cv2.GaussianBlur(unida, (5, 5), 0)
    circulos = cv2.HoughCircles(
        suave, cv2.HOUGH_GRADIENT, dp=1.2, minDist=40,
        param1=120, param2=18, minRadius=radio_min, maxRadius=radio_max)
    if circulos is None:
        return None
    mejor, mejor_puntuacion = None, -1.0
    for x, y, radio in circulos[0]:
        soporte = soporte_anillo(mascara, x, y, radio)
        if soporte < soporte_min:
            continue
        # la esfera es mayor que el trozo que la sigue, asi que peso el radio
        puntuacion = soporte * radio * radio
        if puntuacion > mejor_puntuacion:
            mejor_puntuacion = puntuacion
            mejor = (float(x), float(y), float(radio))
    return mejor


def detectar_esfera(gris, fondo, metodo, umbral_diff):
    mascara = mascara_movimiento(gris, fondo, umbral_diff)
    if metodo == "disco":
        resultado = detectar_disco(mascara)
        if resultado is None:
            # en el ultimo frame la esfera se solapa con la pared por la
            # perspectiva; Hough la encuentra igual por el borde redondo
            resultado = detectar_anillo(mascara)
    else:
        resultado = detectar_anillo(mascara)
    return resultado, mascara


def plantilla_anillo(radio):
    lado = 2 * radio + 1
    plantilla = np.zeros((lado, lado), np.float32)
    cv2.circle(plantilla, (radio, radio), radio, 1.0, grosor_anillo)
    return plantilla


def encajar_radio_fijo(mascara, radio, x_inf, x_sup, cy_previsto):
    # se desliza un anillo del radio conocido por la franja permitida y se queda
    # donde mas blanco toca; la franja respeta el sentido del movimiento
    if x_sup <= x_inf:
        return None
    plantilla = plantilla_anillo(radio)
    mapa = cv2.filter2D((mascara > 0).astype(np.float32), -1, plantilla)
    alto, ancho = mascara.shape
    x0 = max(0, int(x_inf)); x1 = min(ancho, int(x_sup) + 1)
    y0 = max(0, int(cy_previsto - ventana_y)); y1 = min(alto, int(cy_previsto + ventana_y) + 1)
    recorte = mapa[y0:y1, x0:x1]
    if recorte.size == 0:
        return None
    fila, columna = np.unravel_index(np.argmax(recorte), recorte.shape)
    cx = x0 + columna; cy = y0 + fila
    soporte = mapa[cy, cx] / plantilla.sum()
    if soporte < soporte_min_refinado:
        return None
    return float(cx), float(cy), float(radio)


def encajar_borde(mascara, radio, cy_tipico):
    # la esfera entra cortada por un lado; el centro queda a un radio del borde interior
    alto, ancho = mascara.shape
    num, etiquetas, stats, centroides = cv2.connectedComponentsWithStats(mascara)
    mejor, mejor_area = None, 0
    for etiqueta in range(1, num):
        x, y, w, h, area = stats[etiqueta]
        if area < area_min:
            continue
        if abs(centroides[etiqueta][1] - cy_tipico) > radio:
            continue
        toca_derecha = (x + w >= ancho - 1); toca_izquierda = (x <= 0)
        if not (toca_derecha or toca_izquierda):
            continue
        if area > mejor_area:
            mejor_area = area
            cx = (x + radio) if toca_derecha else (x + w) - radio
            mejor = (float(cx), float(cy_tipico), float(radio))
    return mejor


def dibujar_resultado(imagen, resultado, etiqueta, refinado=False):
    anotada = imagen.copy()
    if resultado is not None:
        x, y, radio = resultado
        color = (0, 200, 255) if refinado else (0, 0, 255)
        cv2.circle(anotada, (int(x), int(y)), int(radio), color, 2)
        cv2.circle(anotada, (int(x), int(y)), 3, (255, 255, 255), -1)
    cv2.putText(anotada, etiqueta, (5, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 200, 255), 1)
    return anotada


def mosaico_debug(gris, mascara, anotada):
    gris_bgr = cv2.cvtColor(gris, cv2.COLOR_GRAY2BGR)
    mascara_bgr = cv2.cvtColor(mascara, cv2.COLOR_GRAY2BGR)
    trio = [cv2.resize(p, None, fx=0.5, fy=0.5) for p in (gris_bgr, mascara_bgr, anotada)]
    return cv2.hconcat(trio)

In [ ]:
# ============================================================
#  Deteccion completa de un ensayo -> trayectoria_px.csv
# ============================================================

def procesar_ensayo(nombre):
    cfg = ensayos[nombre]
    frame_ini, frame_fin = cfg["frame_ini"], cfg["frame_fin"]
    metodo, umbral_diff = cfg["metodo"], cfg["umbral"]
    ruta_frames = os.path.join(RUTA_ENTRADA, "Test_" + nombre)
    ruta_salida = os.path.join(RUTA_SALIDA, nombre)

    ruta_anotados = os.path.join(ruta_salida, "anotados")
    ruta_mascaras = os.path.join(ruta_salida, "mascaras")
    ruta_debug = os.path.join(ruta_salida, "debug")
    for carpeta in (ruta_salida, ruta_anotados, ruta_mascaras, ruta_debug):
        os.makedirs(carpeta, exist_ok=True)

    seleccion = cargar_frames(ruta_frames, frame_ini, frame_fin)
    print("{}: {} fotogramas de vuelo libre [{}-{}], metodo '{}'".format(
        nombre, len(seleccion), frame_ini, frame_fin, metodo))
    fondo = construir_fondo(ruta_frames, seleccion)
    cv2.imwrite(os.path.join(ruta_salida, "fondo_mediana.png"), fondo)

    # --- Primera pasada ---
    registros = []
    for numero, archivo in seleccion:
        imagen = cv2.imread(os.path.join(ruta_frames, archivo))
        gris = cv2.cvtColor(imagen, cv2.COLOR_BGR2GRAY)
        resultado, _ = detectar_esfera(gris, fondo, metodo, umbral_diff)
        if resultado is not None:
            x, y, radio = resultado
            registros.append([numero, x, y, radio, 1, 0])
        else:
            registros.append([numero, np.nan, np.nan, np.nan, 0, 0])
    alto, ancho = gris.shape

    # --- Segunda pasada: frames de borde y flojos con radio fijo ---
    if refinar:
        limpios = [r for r in registros
                   if r[4] == 1 and radio_min <= r[3] <= radio_max
                   and r[1] - r[3] > 0 and r[1] + r[3] < ancho]
        radio_fijo = int(round(np.median([r[3] for r in limpios])))
        cy_tipico = float(np.median([r[2] for r in limpios]))
        frames_limpios = np.array([r[0] for r in limpios], dtype=float)
        cx_limpios = np.array([r[1] for r in limpios], dtype=float)
        decreciente = cx_limpios[-1] < cx_limpios[0]
        salto_min = 2.0

        cx_anterior = None
        for registro in registros:
            numero, x, y, radio, detectado, _ = registro
            en_borde = detectado == 1 and (x - radio <= 0 or x + radio >= ancho)
            es_limpio = (detectado == 1 and not en_borde
                         and radio_min <= radio <= radio_max
                         and x - radio > 0 and x + radio < ancho)
            if es_limpio:
                cx_anterior = x
                continue
            posteriores = cx_limpios[frames_limpios > numero]
            if decreciente:
                tope_alto = (cx_anterior - salto_min) if cx_anterior is not None else (ancho + radio_fijo)
                tope_bajo = (posteriores[0] + salto_min) if len(posteriores) else 0.0
            else:
                tope_bajo = (cx_anterior + salto_min) if cx_anterior is not None else 0.0
                tope_alto = (posteriores[0] - salto_min) if len(posteriores) else (ancho + radio_fijo)
            imagen = cv2.imread(os.path.join(ruta_frames, "frame_{:04d}.png".format(numero)))
            gris = cv2.cvtColor(imagen, cv2.COLOR_BGR2GRAY)
            mascara = mascara_movimiento(gris, fondo, umbral_diff)
            if en_borde:
                nuevo = encajar_borde(mascara, radio_fijo, cy_tipico)
                if nuevo is not None and not (tope_bajo <= nuevo[0] <= tope_alto):
                    nuevo = None
            else:
                cx_previsto = float(np.interp(numero, frames_limpios, cx_limpios))
                x_inf = max(tope_bajo, cx_previsto - ventana_x)
                x_sup = min(tope_alto, cx_previsto + ventana_x)
                nuevo = encajar_radio_fijo(mascara, radio_fijo, x_inf, x_sup, cy_tipico)
            if nuevo is not None:
                registro[1], registro[2], registro[3] = nuevo
                registro[4] = 1
                registro[5] = 1
                cx_anterior = nuevo[0]

    # --- Guardar cada fotograma con el resultado final ---
    n_detectados = 0
    for numero, x, y, radio, detectado, refinado_f in registros:
        imagen = cv2.imread(os.path.join(ruta_frames, "frame_{:04d}.png".format(numero)))
        gris = cv2.cvtColor(imagen, cv2.COLOR_BGR2GRAY)
        mascara = mascara_movimiento(gris, fondo, umbral_diff)
        if detectado:
            n_detectados += 1
            resultado = (x, y, radio)
            marca = " (refinado)" if refinado_f else ""
            etiqueta = "f{:04d}  ({:.1f}, {:.1f})  r={:.1f}{}".format(numero, x, y, radio, marca)
        else:
            resultado = None
            etiqueta = "f{:04d}  SIN DETECCION".format(numero)
        anotada = dibujar_resultado(imagen, resultado, etiqueta, refinado_f == 1)
        cv2.imwrite(os.path.join(ruta_anotados, "frame_{:04d}.png".format(numero)), anotada)
        cv2.imwrite(os.path.join(ruta_mascaras, "frame_{:04d}.png".format(numero)), mascara)
        cv2.imwrite(os.path.join(ruta_debug, "frame_{:04d}.png".format(numero)),
                    mosaico_debug(gris, mascara, anotada))

    # --- Guardar la trayectoria en pixeles ---
    ruta_csv = os.path.join(ruta_salida, "trayectoria_px.csv")
    with open(ruta_csv, "w") as f:
        f.write("frame,cx_px,cy_px,r_px,detectado,refinado\n")
        for numero, x, y, radio, detectado, refinado_f in registros:
            if detectado:
                f.write("{},{:.3f},{:.3f},{:.3f},{},{}\n".format(numero, x, y, radio, detectado, refinado_f))
            else:
                f.write("{},,,,{},{}\n".format(numero, detectado, refinado_f))
    print("  detectados {}/{} fotogramas  ->  {}".format(n_detectados, len(registros), ruta_csv))
    return ruta_csv

In [ ]:
for nombre in ensayos:
    procesar_ensayo(nombre)

In [ ]:
# ============================================================
#  Velocidad a partir de la trayectoria en pixeles
# ============================================================

def cargar_csv(nombre):
    ruta = os.path.join(RUTA_SALIDA, nombre, "trayectoria_px.csv")
    frames, cx, cy, r, refinado = [], [], [], [], []
    with open(ruta) as f:
        next(f)
        for linea in f:
            partes = linea.strip().split(",")
            if len(partes) < 5 or partes[4] != "1":
                continue
            frames.append(int(partes[0]))
            cx.append(float(partes[1]))
            cy.append(float(partes[2]))
            r.append(float(partes[3]))
            refinado.append(int(partes[5]) if len(partes) > 5 else 0)
    return tuple(np.array(v) for v in (frames, cx, cy, r, refinado))


def ajuste_robusto(t, y):
    # ajusta una recta descartando de forma iterativa los puntos que se
    # alejan mas de k_outlier desviaciones tipicas del ajuste
    mascara = np.ones(len(t), dtype=bool)
    for _ in range(5):
        m, b = np.polyfit(t[mascara], y[mascara], 1)
        residuo = y - (m * t + b)
        sigma = residuo[mascara].std()
        if sigma == 0:
            break
        nueva = np.abs(residuo) < k_outlier * sigma
        if nueva.sum() == mascara.sum():
            mascara = nueva
            break
        mascara = nueva
    m, b = np.polyfit(t[mascara], y[mascara], 1)
    estimado = m * t[mascara] + b
    ss_res = np.sum((y[mascara] - estimado) ** 2)
    ss_tot = np.sum((y[mascara] - y[mascara].mean()) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 1.0
    return m, b, r2


def medir_velocidad(nombre):
    frames, cx, cy, r, _ = cargar_csv(nombre)
    fiable = (r >= radio_min_vel) & (r <= radio_max_vel) & (cx - r > 0) & (cx + r < ancho_imagen)
    t = frames[fiable].astype(float)
    mx, bx, r2x = ajuste_robusto(t, cx[fiable])
    my, by, r2y = ajuste_robusto(t, cy[fiable])
    v_px = np.hypot(mx, my)
    diametro_px = 2.0 * r[fiable].mean()
    escala = diametro_real_mm / diametro_px       # mm por pixel
    s_frame = v_px * escala                        # mm entre fotogramas
    v_ms = s_frame * fps_real / 1000.0             # m/s
    v_ref = referencia[nombre]
    error = 100.0 * (v_ms - v_ref) / v_ref
    print("ENSAYO {}".format(nombre))
    print("  v imagen   : {:.3f} px/frame  (R2 = {:.5f})".format(v_px, r2x))
    print("  diametro   : {:.2f} px  ->  escala {:.4f} mm/px".format(diametro_px, escala))
    print("  v medida   : {:.2f} m/s   (ref {:.0f}, error {:+.2f} %)".format(v_ms, v_ref, error))
    print("-" * 60)
    return {"v_px": v_px, "diametro_px": diametro_px, "escala": escala, "v_ms": v_ms}


resultados = {nombre: medir_velocidad(nombre) for nombre in ensayos}

In [ ]:
# ============================================================
#  Escala propia (adoptada) vs escala comun del caucho
# ============================================================
# El metodo adoptado usa la escala propia de cada esfera. A modo de
# comprobacion, si se toma el caucho como referencia de 30 mm y se aplica su
# escala a las dos, el diametro del hielo cuadra con su velocidad de referencia.

escala_comun = resultados["Caucho"]["escala"]
print("Escala comun (caucho): {:.4f} mm/px\n".format(escala_comun))
for nombre in ensayos:
    v_px = resultados[nombre]["v_px"]
    v_propia = resultados[nombre]["v_ms"]
    v_comun = v_px * escala_comun * fps_real / 1000.0
    v_ref = referencia[nombre]
    print("{}:  propia {:.2f} m/s ({:+.2f} %)   comun {:.2f} m/s ({:+.2f} %)   ref {:.0f}".format(
        nombre, v_propia, 100 * (v_propia - v_ref) / v_ref,
        v_comun, 100 * (v_comun - v_ref) / v_ref, v_ref))

In [ ]:
# ============================================================
#  Graficas de trayectoria y ajuste
# ============================================================

def grafica_trayectoria(nombre):
    frames, cx, cy, r, refinado = cargar_csv(nombre)
    fiable = (r >= radio_min_vel) & (r <= radio_max_vel) & (cx - r > 0) & (cx + r < ancho_imagen)
    t = frames[fiable].astype(float)
    cx_f, cy_f, refinado_f = cx[fiable], cy[fiable], refinado[fiable]
    mx, bx, _ = ajuste_robusto(t, cx_f)
    my, by, _ = ajuste_robusto(t, cy_f)
    v_px = np.hypot(mx, my)
    base = refinado_f == 0
    extra = refinado_f == 1

    fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
    ax[0].scatter(cx[~fiable], cy[~fiable], c="lightgray", s=20, label="descartado")
    ax[0].scatter(cx_f[base], cy_f[base], c="#1f77b4", s=25, label="deteccion")
    ax[0].scatter(cx_f[extra], cy_f[extra], c="orange", s=25, label="refinado")
    ax[0].set_xlim(0, ancho_imagen); ax[0].set_ylim(ancho_imagen, 0)
    ax[0].set_aspect("equal")
    ax[0].set_xlabel("x [px]"); ax[0].set_ylabel("y [px]")
    ax[0].set_title("Trayectoria en la imagen")
    ax[0].legend(fontsize=8); ax[0].grid(alpha=0.3)

    ax[1].scatter(frames[~fiable], cx[~fiable], c="lightgray", s=20, label="descartado")
    ax[1].scatter(t[base], cx_f[base], c="#1f77b4", s=25, label="deteccion")
    ax[1].scatter(t[extra], cx_f[extra], c="orange", s=25, label="refinado")
    ax[1].plot(t, mx * t + bx, "k-", lw=1.5, label="ajuste: {:+.3f} px/frame".format(mx))
    ax[1].set_xlabel("fotograma"); ax[1].set_ylabel("x [px]")
    ax[1].set_title("x(frame)  ->  v = {:.3f} px/frame".format(v_px))
    ax[1].legend(fontsize=8); ax[1].grid(alpha=0.3)

    fig.suptitle("Ensayo {}  |  v = {:.3f} px/frame".format(nombre, v_px))
    fig.tight_layout()
    salida = os.path.join(RUTA_SALIDA, nombre, "trayectoria_ajuste.png")
    fig.savefig(salida, dpi=120)
    plt.show()
    print("Guardada:", salida)


for nombre in ensayos:
    grafica_trayectoria(nombre)